# 08 — Historical Similar-Event Retrieval for FlightRescue AI

This notebook turns the event-level dataset from Notebook 07 into a **historical analog retrieval system**.

Given a weather/disruption event, the system retrieves the most similar historical Hawaii/OGG episodes and summarizes what happened operationally: cancellations, severe disruption, and observed recovery time.

This is retrieval/decision support rather than a causal claim. Recovery remains the BTS-derived operational proxy defined in Notebook 07.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'data').exists() else cwd.parent
EVENT_FILE = ROOT / 'data/processed/ogg_weather_event_recovery_2020_2025.csv'
OUT_FILE = ROOT / 'data/processed/ogg_event_similarity_index_2020_2025.csv'
print(EVENT_FILE, EVENT_FILE.exists())


## 1. Load historical event episodes


In [ ]:
events = pd.read_csv(EVENT_FILE, low_memory=False)
for c in ['start_dt','end_dt','recovery_dt']:
    if c in events.columns:
        events[c] = pd.to_datetime(events[c], errors='coerce')
print('Events:', events.shape)
display(events.head())


## 2. Define retrieval features

We intentionally exclude outcome fields such as recovery time from similarity distance. Similarity should be based on the event/context, then historical outcomes are displayed **after** retrieval.


In [ ]:
outcome_tokens = [
    'recovery', 'recovered', 'event_cancelled', 'event_delayed15', 'event_severe',
    'event_cancel_rate', 'event_severe_rate', 'peak_hour_cancel_rate',
    'peak_hour_severe_rate'
]
id_tokens = ['event_id','year','storm_records']

numeric_cols = events.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [
    c for c in numeric_cols
    if c not in id_tokens and not any(tok in c for tok in outcome_tokens)
]

# Keep useful context and weather descriptors.
preferred = []
for c in candidate_features:
    if (
        c in ['duration_hours','month','pre_scheduled','baseline_cancel_rate','baseline_severe_rate']
        or c.startswith('event_')
    ):
        preferred.append(c)

retrieval_features = preferred if preferred else candidate_features
print('Retrieval features:', len(retrieval_features))
print(retrieval_features)


## 3. Build standardized historical event vectors


In [ ]:
X = events[retrieval_features].apply(pd.to_numeric, errors='coerce')
usable = [c for c in X.columns if X[c].notna().sum() >= max(5, int(0.20*len(X))) and X[c].nunique(dropna=True) > 1]
X = X[usable]
retrieval_features = usable

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)

print('Usable retrieval features:', len(retrieval_features))
print('Vector matrix:', X_scaled.shape)


## 4. Event-type similarity
Numeric weather/context similarity is combined with a simple Jaccard similarity over NOAA event-type labels. This lets a Flash Flood/High Wind episode receive extra credit when compared with another event containing the same hazards.


In [ ]:
def event_type_set(value):
    if pd.isna(value):
        return set()
    return {x.strip().lower() for x in str(value).split('|') if x.strip()}

def jaccard(a, b):
    a, b = event_type_set(a), event_type_set(b)
    if not a and not b:
        return 1.0
    union = a | b
    return len(a & b)/len(union) if union else 0.0


## 5. Historical analog retrieval function

The default score is 80% standardized numeric cosine similarity + 20% event-type similarity. The query event itself is excluded.


In [ ]:
def retrieve_similar_events(event_id, k=5, numeric_weight=0.80, type_weight=0.20):
    matches = events.index[events['event_id'].eq(event_id)].tolist()
    if not matches:
        raise KeyError(f'Unknown event_id: {event_id}')
    qidx = matches[0]
    numeric_sim = cosine_similarity(X_scaled[qidx:qidx+1], X_scaled).ravel()
    type_sim = np.array([jaccard(events.loc[qidx, 'event_types'], x) for x in events['event_types']])
    score = numeric_weight*numeric_sim + type_weight*type_sim
    score[qidx] = -np.inf
    top = np.argsort(score)[::-1][:k]

    cols = [c for c in [
        'event_id','start_dt','end_dt','event_types','duration_hours',
        'event_scheduled','event_cancel_rate','event_severe_rate',
        'recovery_hours_after_event','recovered_within_72h'
    ] if c in events.columns]
    result = events.loc[top, cols].copy()
    result.insert(1, 'similarity_score', score[top])
    result.insert(2, 'numeric_similarity', numeric_sim[top])
    result.insert(3, 'event_type_similarity', type_sim[top])
    return result.reset_index(drop=True)


## 6. Demonstrate retrieval on a recent historical event


In [ ]:
# Prefer a 2025 event with meaningful operational disruption for demonstration.
demo_pool = events[events['year'].eq(2025)].copy() if 'year' in events.columns else events.copy()
if 'event_severe_rate' in demo_pool.columns and demo_pool['event_severe_rate'].notna().any():
    demo_id = demo_pool.sort_values('event_severe_rate', ascending=False).iloc[0]['event_id']
else:
    demo_id = demo_pool.iloc[-1]['event_id']

print('Query event:', demo_id)
query_cols = [c for c in ['event_id','start_dt','event_types','duration_hours','event_cancel_rate','event_severe_rate','recovery_hours_after_event'] if c in events.columns]
display(events.loc[events['event_id'].eq(demo_id), query_cols])
display(retrieve_similar_events(demo_id, k=5).round(3))


## 7. Leave-one-event-out retrieval diagnostics
We inspect whether closer analogs have more similar recovery times. This is not a full predictive validation yet; it is a sanity check for the retrieval representation.


In [ ]:
diag = []
for _, row in events.iterrows():
    if pd.isna(row.get('recovery_hours_after_event')):
        continue
    nbrs = retrieve_similar_events(row['event_id'], k=min(5, len(events)-1))
    nbrs = nbrs[nbrs['recovery_hours_after_event'].notna()]
    if nbrs.empty:
        continue
    pred = nbrs['recovery_hours_after_event'].median()
    diag.append({
        'event_id': row['event_id'],
        'actual_recovery_h': row['recovery_hours_after_event'],
        'analog_median_recovery_h': pred,
        'absolute_error_h': abs(pred-row['recovery_hours_after_event']),
        'mean_top_similarity': nbrs['similarity_score'].mean(),
    })
diag = pd.DataFrame(diag)
display(diag.describe().round(3))
if not diag.empty:
    print('Median absolute recovery error (hours):', round(diag['absolute_error_h'].median(), 3))
    print('Mean absolute recovery error (hours):', round(diag['absolute_error_h'].mean(), 3))


## 8. Build a reusable similarity index
For the app/API, we save each historical event together with its standardized retrieval vector.


In [ ]:
index_df = events[[c for c in ['event_id','start_dt','end_dt','event_types','event_cancel_rate','event_severe_rate','recovery_hours_after_event'] if c in events.columns]].copy()
for j, c in enumerate(retrieval_features):
    index_df[f'z__{c}'] = X_scaled[:, j]
index_df.to_csv(OUT_FILE, index=False)
print('Saved:', OUT_FILE)
print('Shape:', index_df.shape)


## 9. Product-facing summary helper
This produces the type of historical evidence that can later feed the FlightRescue AI UI.


In [ ]:
def summarize_analogs(event_id, k=5):
    nbrs = retrieve_similar_events(event_id, k=k)
    recovered = nbrs['recovery_hours_after_event'].dropna() if 'recovery_hours_after_event' in nbrs else pd.Series(dtype=float)
    cancel = nbrs['event_cancel_rate'].dropna() if 'event_cancel_rate' in nbrs else pd.Series(dtype=float)
    severe = nbrs['event_severe_rate'].dropna() if 'event_severe_rate' in nbrs else pd.Series(dtype=float)
    return {
        'query_event': event_id,
        'analogs_used': len(nbrs),
        'median_similarity': float(nbrs['similarity_score'].median()),
        'historical_median_cancel_rate': float(cancel.median()) if len(cancel) else None,
        'historical_median_severe_rate': float(severe.median()) if len(severe) else None,
        'historical_median_recovery_hours': float(recovered.median()) if len(recovered) else None,
        'historical_recovery_range_hours': [float(recovered.min()), float(recovered.max())] if len(recovered) else None,
    }

summary = summarize_analogs(demo_id, k=5)
print(json.dumps(summary, indent=2))


## Next step
Notebook 09 can connect the two branches of FlightRescue AI:

1. **flight-level disruption probability** from Notebook 06;
2. **historical analogs and recovery evidence** from Notebook 08.

That combined inference layer can produce a passenger-facing risk/recovery response suitable for the later API and GitHub Pages UI.
